# 09 · Ranking & Retrieval Evaluation

## What this notebook covers
Retrieval systems (search engines, RAG pipelines, recommenders) require different metrics than classifiers. A retrieved document is not simply "correct" or "incorrect" — position matters, and relevance is often graded. This notebook implements the full standard retrieval metric suite and shows how to build a `RetrievalEvaluator` for RAG pipeline benchmarking.

## The retrieval metrics menu
| Metric | What it rewards | Notes |
|--------|----------------|-------|
| **Precision@k** | Fraction of top-k that are relevant | Ignores recall |
| **Recall@k** | Fraction of all relevant docs in top-k | Ignores position |
| **MRR** | Reciprocal rank of first relevant doc | Good for "find one good result" |
| **NDCG@k** | Graded relevance, position-discounted | Standard for multi-grade relevance |

## Why this matters for LLM systems
RAG pipelines are only as good as their retrieval step. Evaluating RAG at the generation level (ROUGE, LLM-judge) without also evaluating retrieval leads to silent degradation when the retriever fails but the LLM hallucinates a plausible answer. Separating retrieval evaluation from generation evaluation is standard practice in production RAG systems.

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Dict
import matplotlib.pyplot as plt
np.random.seed(42)

@dataclass
class RetrievalResult:
    query_id: str
    retrieved_ids: List[str]        # ordered list of retrieved doc IDs
    relevant_ids: List[str]         # ground-truth relevant doc IDs (binary)
    relevance_grades: Dict[str, int] = field(default_factory=dict)  # doc_id -> grade (0-3)

def precision_at_k(result: RetrievalResult, k: int) -> float:
    top_k = result.retrieved_ids[:k]
    hits = sum(1 for doc in top_k if doc in result.relevant_ids)
    return hits / k

def recall_at_k(result: RetrievalResult, k: int) -> float:
    if not result.relevant_ids:
        return 0.0
    top_k = result.retrieved_ids[:k]
    hits = sum(1 for doc in top_k if doc in result.relevant_ids)
    return hits / len(result.relevant_ids)

def reciprocal_rank(result: RetrievalResult) -> float:
    for i, doc in enumerate(result.retrieved_ids, 1):
        if doc in result.relevant_ids:
            return 1.0 / i
    return 0.0

def dcg_at_k(result: RetrievalResult, k: int) -> float:
    """Discounted Cumulative Gain with graded relevance."""
    score = 0.0
    for i, doc in enumerate(result.retrieved_ids[:k], 1):
        grade = result.relevance_grades.get(doc, (1 if doc in result.relevant_ids else 0))
        score += (2 ** grade - 1) / np.log2(i + 1)
    return score

def ndcg_at_k(result: RetrievalResult, k: int) -> float:
    """Normalised DCG: DCG / ideal DCG."""
    actual_dcg = dcg_at_k(result, k)
    ideal_grades = sorted(result.relevance_grades.values(), reverse=True)[:k]
    if not ideal_grades:
        ideal_grades = [1] * min(len(result.relevant_ids), k)
    ideal_dcg = sum(
        (2 ** g - 1) / np.log2(i + 1)
        for i, g in enumerate(ideal_grades, 1)
    )
    return actual_dcg / ideal_dcg if ideal_dcg > 0 else 0.0

print("Metrics defined: P@k, R@k, MRR, DCG@k, NDCG@k")

In [ ]:
class RetrievalEvaluator:
    def __init__(self, k_values=(1, 3, 5, 10)):
        self.k_values = k_values

    def evaluate(self, results: List[RetrievalResult]) -> pd.DataFrame:
        rows = []
        for r in results:
            row = {"query_id": r.query_id, "mrr": round(reciprocal_rank(r), 4)}
            for k in self.k_values:
                row[f"p@{k}"]    = round(precision_at_k(r, k), 4)
                row[f"r@{k}"]    = round(recall_at_k(r, k), 4)
                row[f"ndcg@{k}"] = round(ndcg_at_k(r, k), 4)
            rows.append(row)
        return pd.DataFrame(rows)

    def summary(self, results: List[RetrievalResult]) -> pd.Series:
        df = self.evaluate(results)
        return df.drop(columns=["query_id"]).mean().round(4)

def build_retrieval_eval_set(n_queries=20, corpus_size=200, avg_relevant=3):
    """Synthetic eval set for a RAG pipeline."""
    corpus = [f"doc_{i:04d}" for i in range(corpus_size)]
    results = []
    for qid in range(n_queries):
        relevant = list(np.random.choice(corpus, avg_relevant, replace=False))
        grades = {d: np.random.randint(1, 4) for d in relevant}
        # Simulate retrieved: relevant docs appear in top-10 with prob 0.6
        retrieved = []
        for d in relevant:
            if np.random.rand() < 0.6:
                retrieved.append(d)
        noise = [d for d in corpus if d not in relevant]
        np.random.shuffle(noise)
        retrieved.extend(noise[:15 - len(retrieved)])
        np.random.shuffle(retrieved)
        results.append(RetrievalResult(
            query_id=f"q_{qid:03d}",
            retrieved_ids=retrieved[:10],
            relevant_ids=relevant,
            relevance_grades=grades,
        ))
    return results

eval_set = build_retrieval_eval_set(n_queries=50)
evaluator = RetrievalEvaluator(k_values=(1, 3, 5, 10))
summary = evaluator.summary(eval_set)
print(summary.to_string())

In [ ]:
# Three-system benchmark: simulate a baseline, improved, and re-ranked system
def simulate_system(n_queries=50, corpus_size=200, hit_prob=0.5):
    corpus = [f"doc_{i:04d}" for i in range(corpus_size)]
    results = []
    for qid in range(n_queries):
        relevant = list(np.random.choice(corpus, 3, replace=False))
        grades = {d: np.random.randint(1, 4) for d in relevant}
        retrieved = [d for d in relevant if np.random.rand() < hit_prob]
        noise = [d for d in corpus if d not in relevant]
        np.random.shuffle(noise)
        retrieved.extend(noise[:10 - len(retrieved)])
        np.random.shuffle(retrieved)
        results.append(RetrievalResult(
            query_id=f"q_{qid:03d}",
            retrieved_ids=retrieved[:10],
            relevant_ids=relevant,
            relevance_grades=grades,
        ))
    return results

systems = {
    "BM25 Baseline"  : simulate_system(hit_prob=0.4),
    "Dense Retriever": simulate_system(hit_prob=0.65),
    "Re-ranker"      : simulate_system(hit_prob=0.80),
}

summary_rows = {}
for name, results in systems.items():
    summary_rows[name] = evaluator.summary(results)

comparison_df = pd.DataFrame(summary_rows).T
print(comparison_df[["mrr", "p@5", "r@5", "ndcg@5", "ndcg@10"]].to_string())

# Bar chart
metrics_to_plot = ["mrr", "p@5", "ndcg@10"]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, metric in zip(axes, metrics_to_plot):
    vals = comparison_df[metric]
    ax.bar(vals.index, vals.values, color=["steelblue", "seagreen", "tomato"])
    ax.set_title(metric.upper())
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(f"{base}/09_retrieval_eval.png", dpi=120)
plt.show()